# Starting point for Instruct model and Activation gathering

In [84]:
# gloabl imports
import yaml
import numpy as np
import pandas as pd
import sys
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from datasets import load_dataset
from tqdm import tqdm
import plotly.express as px
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA


# Local imports
sys.path.append('../src')
from detection import *
from ml_model import *
from llm_wrapper import *

### Load model

Using LLM_wrapper

In [20]:
model_id = "meta-llama/Meta-Llama-3-8B"

with open("../param.yaml", 'r') as file:
    params = yaml.safe_load(file)

print("model_parameters:", params["model_arguments"])
params["model_arguments"]["model_id"] = model_id

try:
    llm
except:
    llm = LLMWrapper(
        hf_token = params["hf_token"],
        **params["model_arguments"]
    )

model_parameters: {'model_id': 'meta-llama/Meta-Llama-3-8B-Instruct', 'load_in_8bit': False, 'torch_dtype': 'torch.bfloat16'}


### Data examples

In [21]:
# nb_samples = 10
# # Load HC3 dataset - wiki_csai subset (Not sure if it has been trained on.. but you get the idea)
# trained_data = load_dataset("Hello-SimpleAI/HC3", 'wiki_csai')
# trained_data = [trained_data['train'][i]['human_answer'] for i in range(len(trained_data['train']))]
trained_data = [
    # 1. Albert Einstein
    "Physicist Albert Einstein revolutionized modern science with his theory of relativity and made foundational contributions to quantum mechanics, becoming one of the most influential thinkers of the 20th century.",
    # 2. Marie Curie
    "Scientist Marie Curie pioneered research on radioactivity, becoming the first person to win two Nobel Prizes and transforming the fields of physics and chemistry.",
    # 3. Nelson Mandela
    "Anti-apartheid leader Nelson Mandela fought for racial equality in South Africa, later becoming the country’s first Black president and a global symbol of peace and justice.",
    # 4. Frida Kahlo
    "Artist Frida Kahlo became world-renowned for her vividly personal and surreal self-portraits, exploring identity, pain, and resilience through a unique visual language.",
    # 5. Leonardo da Vinci
    "Polymath Leonardo da Vinci mastered art, engineering, anatomy, and invention, leaving behind timeless works like the Mona Lisa while embodying the spirit of the Renaissance.",
    # 6. Martin Luther King Jr.
    "Civil rights leader Martin Luther King Jr. advocated for justice and equality through nonviolent protest, delivering iconic speeches that helped reshape American history.",
    # 7. Malala Yousafzai
    "Activist Malala Yousafzai became a global voice for girls’ education after surviving an assassination attempt, later becoming the youngest-ever Nobel Peace Prize laureate.",
    # 8. Stephen Hawking
    "Theoretical physicist Stephen Hawking advanced our understanding of black holes and cosmology, while inspiring millions with his perseverance and bestselling scientific writings.",
    # 9. Beyoncé Knowles
    "Performer Beyoncé Knowles rose to global stardom through her powerful vocals, groundbreaking visual albums, and cultural influence across music, fashion, and activism.",
    # 10. Mahatma Gandhi
    "Leader Mahatma Gandhi championed nonviolent resistance in the struggle for India’s independence, leaving a lasting legacy as an icon of civil disobedience and moral leadership."
]

In [22]:
# Create some random untrained data (just for testing)
untrained_data = [
    # 1. Dr. Liora Vensel
    "Cognitive botanist Liora Vensel is known for studying plant-to-plant electrical communication. Her discovery of the “Silent Orchard,” a forest that reacts emotionally to visitors, made her a leading figure at the Institute of Neuroflora.",
    # 2. Mateo Raskov
    "Street photographer Mateo Raskov captures fleeting acts of humanity in sprawling megacities. His award-winning exhibition The Quiet Crowd cemented his reputation as a master of unnoticed moments.",
    #3. Anika Duvall
    "Underwater architect Anika Duvall designs coral-based habitats that grow and evolve like living cities. After a childhood on a floating commune, she founded Oceanframe Labs to fight coastline degradation.",
    #4. Idris Jang
    "Robotics ethicist Idris Jang mediates conflicts between AI engineers and governments. He’s best known for co-authoring the Global Compassionate Robotics Charter, promoting “empathetic automation.”",
    # 5. Sera Calloway
    "Former freight pilot Sera Calloway survived a catastrophic gyre storm and now trains extreme-weather rescue teams. Her memoir Winds That Remember is widely used in climate-risk education.",
    # 6. Prof. Yaroslav Finch   
    "Mathematician Yaroslav Finch discovered a new fractal pattern while analyzing café background noise. His quirky habit of carrying tuning forks everywhere has become an unofficial trademark.",
    # 7. Keiko Marenzi
    "Game developer Keiko Marenzi created Starling Streets, an award-winning narrative game where constellations roam city alleys. She advocates for neurodiversity in creative tech fields.",  
    # 8. Amadou “Mads” Keita
    "Solar engineer Amadou Keita, known as “Mads,” invented the portable “Sunspine” light tower that can power entire rural communities. He spends most of his year traveling to install his systems firsthand.",
    # 9. Thalia Groves
    "Poet Thalia Groves writes personalized verses on biodegradable leaves in the floating markets of New Lyra. She’s rumored to be the anonymous author behind the viral poetry collection Leaves for Strangers.",
    # 10. Orion Hale
    "Neuroscientist Orion Hale began as a stage illusionist before turning to the science of perception. He now collaborates with artists to create exhibitions that challenge how people interpret reality."
]

### Normal generation code

In [23]:
print("Gathering arguments:", params["gathering_arguments"])

Gathering arguments: {'gathering_layers': [15], 'max_token_seq': 200}


In [98]:
bsz = 1

def run_gathering_data_pipeline(input_texts, llm, params):
    dict_list = []
    # Same parameters as generation, but max_new_tokens = 1. Temp, top_k, top_p, etc have no impact on the first activations, but are still set-up for consistency.
    gathering_kwargs = params["generation_arguments"].copy()
    gathering_kwargs.pop("generation_batch_size")
    gathering_kwargs["max_new_tokens"] = 1

    # Registering the hooks to gather the activations
    saving_hooks = llm.register_hooks("gather", params["gathering_arguments"]["gathering_layers"])

    for input_text in tqdm(input_texts):        
        # print("Processing input text:", input_text)
        # Run the model to gather activations
        _ = llm.pipeline(input_text, **gathering_kwargs)

        # Collect activations from hooks
        activations = {}
        for hook in saving_hooks:
            for layer in params["gathering_arguments"]["gathering_layers"]:
                if hook.layer_name == layer:
                    activations[layer] = hook.activations

        dict_list.append({
            "input_text": input_text,
            "activations": activations
        })

    # Remove hooks after gathering
    for hook in saving_hooks:
        hook.remove()


    return dict_list

In [100]:
trained_activations = run_gathering_data_pipeline(trained_data, llm, params)
untrained_activations = run_gathering_data_pipeline(untrained_data, llm, params)

100%|██████████| 10/10 [00:00<00:00, 26.70it/s]


In [95]:
print("Shape of the activations at a given layer [Sequence length, Hidden dimention] :", trained_activations[0]["activations"][15].shape)     

Shape of the activations at a given layer [Sequence length, Hidden dimention] : (37, 4096)


In [96]:
# train a LDA on half of the sentences, and test with the other half
half_index = len(trained_activations) // 2

min_seq_length = min(
    min([data["activations"][15].shape[0] for data in trained_activations]),
    min([data["activations"][15].shape[0] for data in untrained_activations])
)

input_text_train = []
input_text_test = []
X_train = []
y_train = []
X_test = []
y_test = []
for i, data in enumerate(trained_activations):
    print("Processing trained data index:", i, "with input text:", data["input_text"])
    # Flatten the activations for the given layer (e.g., layer 15)
    layer_activations = data["activations"][15][-1]  # Shape: [Sequence length, Hidden dimension]
    layer_activations = [layer_activations]

    if i < half_index:
        X_train.extend(layer_activations)
        y_train.extend([1] * len(layer_activations))  # Label 1 for trained data
        input_text_train.extend([data["input_text"]] * len(layer_activations))
    else:
        X_test.extend(layer_activations)
        y_test.extend([1] * len(layer_activations))  # Label 1 for trained data
        input_text_test.extend([data["input_text"]] * len(layer_activations))
        
for i, data in enumerate(untrained_activations):
    # Flatten the activations for the given layer (e.g., layer 15)
    layer_activations = data["activations"][15][-1]  # Shape: [Sequence length, Hidden dimension]
    layer_activations = [layer_activations]
    
    if i < half_index:
        X_train.extend(layer_activations)
        y_train.extend([0] * len(layer_activations))  # Label 0 for untrained data
        input_text_train.extend([data["input_text"]] * len(layer_activations))
    else:
        X_test.extend(layer_activations)
        y_test.extend([0] * len(layer_activations))  # Label 0 for untrained data
        input_text_test.extend([data["input_text"]] * len(layer_activations))

X_train = np.array(X_train)
y_train = np.array(y_train)
X_test = np.array(X_test)
y_test = np.array(y_test)
lda = LDA()
lda.fit(X_train, y_train)
accuracy = lda.score(X_test, y_test)
print(f"LDA classification accuracy: {accuracy * 100:.2f}%")
#!/usr/bin/env python3


Processing trained data index: 0 with input text: Physicist Albert Einstein revolutionized modern science with his theory of relativity and made foundational contributions to quantum mechanics, becoming one of the most influential thinkers of the 20th century.
Processing trained data index: 1 with input text: Scientist Marie Curie pioneered research on radioactivity, becoming the first person to win two Nobel Prizes and transforming the fields of physics and chemistry.
Processing trained data index: 2 with input text: Anti-apartheid leader Nelson Mandela fought for racial equality in South Africa, later becoming the country’s first Black president and a global symbol of peace and justice.
Processing trained data index: 3 with input text: Artist Frida Kahlo became world-renowned for her vividly personal and surreal self-portraits, exploring identity, pain, and resilience through a unique visual language.
Processing trained data index: 4 with input text: Polymath Leonardo da Vinci master

In [97]:
# Project on 2d with a PCA for visualization with binary labels
all_activations = X_train.tolist() + X_test.tolist()
all_labels = y_train.tolist() + y_test.tolist()
all_input_texts = input_text_train + input_text_test

print("X_train shape:", X_train.shape)
print("all labelas:", all_labels)
print("All input texts:", len(all_input_texts), all_input_texts)

pca = PCA(n_components=2)
all_activations_2d = pca.fit_transform(all_activations)
# plot with plotly
fig = px.scatter(
    x=all_activations_2d[:, 0],
    y=all_activations_2d[:, 1],
    color=[str(label) for label in all_labels],
    # add the input text as hover data
    hover_data={'input_text': input_text_train + input_text_test},
    labels={'color': 'Trained (1) vs Untrained (0)'},
    title='PCA of Activations at Layer 15'
)
fig.show()

X_train shape: (10, 4096)
all labelas: [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0]
All input texts: 20 ['Physicist Albert Einstein revolutionized modern science with his theory of relativity and made foundational contributions to quantum mechanics, becoming one of the most influential thinkers of the 20th century.', 'Scientist Marie Curie pioneered research on radioactivity, becoming the first person to win two Nobel Prizes and transforming the fields of physics and chemistry.', 'Anti-apartheid leader Nelson Mandela fought for racial equality in South Africa, later becoming the country’s first Black president and a global symbol of peace and justice.', 'Artist Frida Kahlo became world-renowned for her vividly personal and surreal self-portraits, exploring identity, pain, and resilience through a unique visual language.', 'Polymath Leonardo da Vinci mastered art, engineering, anatomy, and invention, leaving behind timeless works like the Mona Lisa while embodying the sp